In [1]:
import pandas as pd
import glob
import numpy as np


In [2]:
belt_path = './grace-0.6m/breathing-belt/*.csv'
sensor_path = './grace-0.6m/sensor/*.csv'

belt_csvs = glob.glob(belt_path)
sensor_csvs = glob.glob(sensor_path)

In [3]:
belt_df_list = [pd.read_csv(path) for path in belt_csvs]
sensor_df_list = [pd.read_csv(path) for path in sensor_csvs]


for i in range(len(sensor_df_list)):
    # initial data cleaning
    to_subtract = sensor_df_list[i].iloc[0,0]
    sensor_df_list[i].rename(columns = {'Timestamp':'Time', 'Breath Rate':'sensor-bpm'}, inplace=True)
    sensor_df_list[i]['Time'] = sensor_df_list[i]['Time'] - to_subtract
    sensor_df_list[i]['Time'] = np.floor(sensor_df_list[i]['Time']/10)*10
    

    # group data by time and use average breath rate
    sensor_df_list[i] = sensor_df_list[i].groupby('Time', as_index=False)['sensor-bpm'].mean()
    time_drop_condition = sensor_df_list[i]["Time"] <= 300
    sensor_df_list[i] = sensor_df_list[i][time_drop_condition]
    # print(sensor_df_list[i].tail())

    # add additional columns to keep track of data
    sensor_df_list[i].insert(loc = 0, column = 'Height', value = f"{i+1}")
    sensor_df_list[i].insert(loc = 1, column="Position", value = "Front")
    sensor_df_list[i].insert(loc=2, column="Distance (m)", value = 0.6)

    # process belt data
    belt_df_list[i].dropna(inplace = True)
    




belt_df = pd.concat(belt_df_list, ignore_index=True)
sensor_df = pd.concat(sensor_df_list, ignore_index=True)

sensor_df.insert(4, 'bpm', belt_df['bpm'])
# sensor_df['bpm'] = belt_df['bpm']

print(sensor_df.head())


  Height Position  Distance (m)  Time        bpm  sensor-bpm
0      1    Front           0.6   0.0  15.350877    4.665901
1      1    Front           0.6  10.0  13.254786    4.498012
2      1    Front           0.6  20.0  13.077594    5.068226
3      1    Front           0.6  30.0  14.383561    4.603927
4      1    Front           0.6  40.0  13.605442    4.310900


In [4]:
# dataframe we will work with for data analysis - no more need for timestamps
df = sensor_df
df.pop(df.columns[3])

# prep sam's data + merge
sam_df = pd.read_csv('./cleanBreathData.csv')

sam_df.pop(sam_df.columns[0])
sam_df.pop(sam_df.columns[-1])
sam_df.pop(sam_df.columns[-1])

sam_df.rename(columns={'height': 'Height', 'position': 'Position', 'breathingBeltRespiration': 'bpm', 'sensorRespiration':'sensor-bpm'}, inplace=True)

sam_df.insert(loc=2, column="Distance (m)", value = 0.6)

df = pd.concat([df,sam_df])
print(df.head())
print(df.tail())

  Height Position  Distance (m)        bpm  sensor-bpm
0      1    Front           0.6  15.350877    4.665901
1      1    Front           0.6  13.254786    4.498012
2      1    Front           0.6  13.077594    5.068226
3      1    Front           0.6  14.383561    4.603927
4      1    Front           0.6  13.605442    4.310900
     Height Position  Distance (m)        bpm  sensor-bpm
1810      3     Side           0.6   8.113590    9.641214
1811      3     Side           0.6  10.006671   12.158156
1812      3     Side           0.6  12.798635   13.101744
1813      3     Side           0.6  14.363885    8.100053
1814      3     Side           0.6  13.473054    8.456598


In [5]:
# add deviation + percent error
df['deviation'] = df['sensor-bpm'] - df['bpm']
df['percent-error'] = (df['sensor-bpm'] - df['bpm'])/df['bpm']

print(df.head())

df.to_csv("./grace-0.6m/cleaned.csv", index=False)

  Height Position  Distance (m)        bpm  sensor-bpm  deviation  \
0      1    Front           0.6  15.350877    4.665901 -10.684975   
1      1    Front           0.6  13.254786    4.498012  -8.756775   
2      1    Front           0.6  13.077594    5.068226  -8.009368   
3      1    Front           0.6  14.383561    4.603927  -9.779634   
4      1    Front           0.6  13.605442    4.310900  -9.294542   

   percent-error  
0      -0.696050  
1      -0.660650  
2      -0.612450  
3      -0.679917  
4      -0.683149  
